# 004 - True Structured Output vs Prompt Engineering

This notebook compares schema-based structured output with prompt engineering for reliable JSON generation.

## Setup

Import libraries and configure the API.

In [ ]:
import google.generativeai as genai
import os
import json
import pandas as pd
from pydantic import BaseModel, Field
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Get API key from environment variable
api_key = os.getenv('GOOGLE_API_KEY')
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)
model = genai.GenerativeModel("models/gemini-2.0-flash-lite")

## Define Data Models and Schemas

Create Pydantic models and JSON schemas for structured output.

In [ ]:
# Define a simple user profile model
class UserProfile(BaseModel):
    username: str = Field(description="Unique username")
    name: str = Field(description="First name")
    last_name: str = Field(description="Last name")
    city: str = Field(description="City name")
    age: int = Field(description="Age between 18-100")

# Simple schema for current API
user_schema = {
    "type": "object",
    "properties": {
        "username": {"type": "string"},
        "name": {"type": "string"},
        "last_name": {"type": "string"},
        "city": {"type": "string"},
        "age": {"type": "integer"}
    },
    "required": ["username", "name", "last_name", "city"]
}

users_list_schema = {
    "type": "array",
    "items": user_schema
}

## Helper Functions

Create functions for structured output and prompt engineering approaches.

In [ ]:
def ask_with_schema(prompt, schema):
    """Ask the model with structured output using a schema"""
    print(f"Prompt: {prompt}")
    print("-" * 60)
    
    response = model.generate_content(
        prompt,
        generation_config=genai.GenerationConfig(
            response_mime_type="application/json",
            response_schema=schema,
        )
    )
    
    print(f"Response:\n{response.text}")
    
    # Parse the JSON response
    data = json.loads(response.text)
    print(f"\n✅ Structured data received!")
    
    return data

## Generate Structured Output

Using schema to guarantee JSON format.

In [ ]:
result1 = ask_with_schema(
    "Generate a user profile",
    user_schema
)

## Generate Multiple Users with Schema

Schema-based generation of multiple records.

In [ ]:
result2 = ask_with_schema(
    "Generate 3 diverse user profiles",
    users_list_schema
)

## Create DataFrame from Results

Process the results into a pandas DataFrame.

In [ ]:
# Create DataFrame from results
all_users = []
for result in [result1, result2]:
    if result:
        if isinstance(result, list):
            all_users.extend(result)
        elif isinstance(result, dict):
            all_users.append(result)

if all_users:
    df = pd.DataFrame(all_users)
    print(f"\n📋 FINAL DATAFRAME ({len(df)} users):")
    print(df.to_string(index=False))
    
    df.to_csv('structured_profiles_simple.csv', index=False)
    print(f"\n💾 Saved to 'structured_profiles_simple.csv'")